# 01 — Data explorationGoal: understand the VCBench public split (4500 founders, 9% success rate)before building any model. We look at class balance, industry distribution,education ranking, and prior exits.

In [ ]:
# Setupimport pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snssns.set_style('whitegrid')df = pd.read_csv('../data/vcbench_final_public.csv')print(f'Rows: {len(df)} | Columns: {list(df.columns)}')df.head()

## Class balance

In [ ]:
print(df['success'].value_counts())print(f"\nSuccess rate: {df['success'].mean()*100:.2f}%")print(f"Imbalance ratio: {(df['success']==0).sum() / max(1,(df['success']==1).sum()):.1f}x")

**Takeaway.** Heavy imbalance (10.1×). Default 0.5 threshold and accuracyare not appropriate; we'll use F0.5 with class weighting.

## Industry distribution

In [ ]:
top10 = df['industry'].value_counts().head(10)print(top10)print(f"\nTop 3 industries cover {top10.head(3).sum()/len(df)*100:.1f}% of the dataset")

**Takeaway.** Tech-dominant: Software Dev + Tech/Internet + IT Services = ~34%.

## Education prestige (QS ranking)

In [ ]:
import astdef safe_parse(s):    if pd.isna(s): return []    try: return ast.literal_eval(s) if isinstance(s, str) else s    except: return []def parse_qs(qs):    if not qs: return None    qs = str(qs).strip()    if qs == '200+': return 250    if '-' in qs:        try:            lo, hi = qs.split('-')            return (int(lo)+int(hi))/2        except: return None    try: return float(qs)    except: return Nonedef best_qs(s):    edus = safe_parse(s)    ranks = [parse_qs(e.get('qs_ranking','')) for e in edus]    ranks = [r for r in ranks if r is not None]    return min(ranks) if ranks else Nonedf['best_qs'] = df['educations_json'].apply(best_qs)def tier(qs):    if qs is None: return 'No ranking'    if qs <= 10: return 'Top 10'    if qs <= 50: return 'Top 11–50'    if qs <= 100: return 'Top 51–100'    if qs <= 200: return 'Top 101–200'    return '200+'df['qs_tier'] = df['best_qs'].apply(tier)stats = df.groupby('qs_tier').agg(n=('success','count'),                                  rate=('success', lambda x: x.mean()*100)).round(1)print(stats)

**Takeaway.** The QS effect is huge: Top 10 = 19.6% success vs 200+ = 6.0%.That's a 3× spread, the strongest single signal in the dataset.

## Prior exits

In [ ]:
df['n_ipos'] = df['ipos'].apply(lambda x: len(safe_parse(x)))df['n_acq']  = df['acquisitions'].apply(lambda x: len(safe_parse(x)))df['n_exits'] = df['n_ipos'] + df['n_acq']df['exit_bucket'] = df['n_exits'].apply(lambda x: '4+' if x>=4 else str(int(x)))exit_stats = df.groupby('exit_bucket').agg(    n=('success','count'),    rate=('success', lambda x: x.mean()*100)).round(1)print(exit_stats.reindex(['0','1','2','3','4+']).dropna())

**Takeaway.** Track-record effect: a serial entrepreneur with 2+ exits is~6× more likely to succeed than a first-timer. We'll encode this as astrong feature.